In [1]:
import pandas as pd
import os
import boto3
import numpy as np
import math
from pprint import pprint
from passwords import *
import pyodbc

try:
    from surgeo import BIFSGModel
except ModuleNotFoundError:
    ! pip install surgeo
    from surgeo import BIFSGModel

try:
    from gender_guesser.detector import Detector
except ModuleNotFoundError:
    ! pip install gender_guesser
    from gender_guesser.detector import Detector

### Functions

In [2]:
def remove_hyphen(str_value):
    try:
        return int(str_value.split('-')[0].split('.')[0])
    except ValueError:
        return 'ERROR'

In [3]:
def get_race(ser_race_prob):
    return ser_race_prob.idxmax()

In [4]:
def get_gender(str_first_name, cls_detector):
    # return
    return cls_detector.get_gender(str_first_name)

In [5]:
def download_from_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

In [6]:
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [7]:
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')

Project: 20231010-gen-xii


### Create output directory

In [8]:
str_dirname_output = './output'
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Connect to DB

In [9]:
# connnect to db
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)

### Read query

In [10]:
# get original query
str_filename = 'query.sql'
str_local_path = f'./sql/{str_filename}'
str_query = open(str_local_path, 'r').read()
print(str_query)

with tblBase as
(
select
    tblAccount.bigAccountId,
    tblAccount.bigDebtorId,
    1 as bitDebtor
from tblAccount
where dtmStampCreation>='10/01/2013'
and dtmStampCreation<='12/31/2019'
union
select
    tblAccount.bigAccountId,
    tblCosigner.bigDebtorId_cosigner as bigDebtorId,
    0 as bitDebtor
from tblAccount
left outer join tblCosigner on tblCosigner.bigAccountId=tblAccount.bigAccountId
where dtmStampCreation>='10/01/2013'
and dtmStampCreation<='12/31/2019'
and tblCosigner.bigDebtorId_cosigner is not null
)

select 
    tblBase.*,
    tblDebtor.strNameFirst,
    tblDebtor.strNameLast,
    tblDebtor.dtmBirthday,
    tblAddress.strZipCode,
    tblDebtor.dtmStampCreation
from tblBase
left outer join 
(
    select
        bigDebtorId,
        min(bigAddressId) as bigAddressId
    from tblAddress
    group by bigDebtorId
)tblMin on tblMin.bigDebtorId=tblBase.bigDebtorId
left join tblAddress on tblAddress.bigAddressId = tblMin.bigAddressId
left outer join tblDebtor on tblDebtor.bigD

### Pull data

In [11]:
%%time

# read query and pull into df
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
list_cols = [
    'bigAccountId',
    'bigDebtorId',
    'strNameFirst',
    'strNameLast',
    'strZipCode',
]
df = df[list_cols]

# show
df

Wall time: 36.5 s


,bigAccountId,bigDebtorId,strNameFirst,strNameLast,strZipCode
0,1338314,1757483,john,doe,84663
1,1338520,1757748,RACHEL,COX,80112
2,1338612,1757865,J.R.,SEXTON,40403
3,1338624,1757881,john,doe,63129
4,1338702,1757981,PAMELA,BILLINGSLEA,30297
...,...,...,...,...,...
2224800,4254698,5447364,CHARLES,MORRIS,84660
2224801,4254704,5447371,STEPHANEY,GONZALEZ,60477
2224802,4254705,5447373,STACIE,KIRKLAND,23604
2224803,4254709,5447380,DANIELL,SABATER,60016


### Clean

In [12]:
%%time

# convert zip to str
df['strZipCode'] = df['strZipCode'].astype(str)

# rm hyphen
df['strZipCode'] = df['strZipCode'].apply(remove_hyphen)

# filter errors
df = df[df['strZipCode'] != 'ERROR']

# set dtype
for col in ['bigAccountId','bigDebtorId','strZipCode']:
    df[col] = df[col].astype(int)

# capitalize names
for col in ['strNameFirst','strNameLast']:
    df[col] = df[col].str.upper()

<timed exec>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Wall time: 4.62 s


### Get race

In [13]:
%%time

# init
cls_detector = BIFSGModel()

# get probability of each race
list_cols = ['white','black','api','native','multiple','hispanic']
df[list_cols] = cls_detector.get_probabilities(
    df['strNameFirst'],
    df['strNameLast'],
    df['strZipCode'],
)[list_cols]

# get max prob
df['max_proba_race'] = df[list_cols].apply(max, axis=1)

# get race
df['race'] = df[list_cols].apply(get_race, axis=1)

# drop
df.drop(list_cols, axis=1, inplace=True)

# show
df

C:\Users\aengland\Anaconda3\lib\site-packages\pandas\core\frame.py:3191: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self[k1] = value[k2]
<timed exec>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Wall time: 2min 54s


C:\Users\aengland\Anaconda3\lib\site-packages\pandas\core\frame.py:4308: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().drop(


,bigAccountId,bigDebtorId,strNameFirst,strNameLast,strZipCode,max_proba_race,race
0,1338314,1757483,JOHN,DOE,84663,0.977112,white
1,1338520,1757748,RACHEL,COX,80112,0.976314,white
2,1338612,1757865,J.R.,SEXTON,40403,NaN,NaN
3,1338624,1757881,JOHN,DOE,63129,0.982650,white
4,1338702,1757981,PAMELA,BILLINGSLEA,30297,0.954750,black
...,...,...,...,...,...,...,...
2224800,4254698,5447364,CHARLES,MORRIS,84660,NaN,NaN
2224801,4254704,5447371,STEPHANEY,GONZALEZ,60477,NaN,NaN
2224802,4254705,5447373,STACIE,KIRKLAND,23604,NaN,NaN
2224803,4254709,5447380,DANIELL,SABATER,60016,NaN,NaN


### Get gender

In [14]:
%%time

# init
cls_detector = Detector()

# capitalize first name
df['strNameFirst_cap'] = df['strNameFirst'].str.capitalize()

# make target
df['gender'] = df.apply(
    lambda x: get_gender(
        str_first_name=x['strNameFirst_cap'], 
        cls_detector=cls_detector,
    ),
    axis=1,
)

# drop
df.drop('strNameFirst_cap', axis=1, inplace=True)

# show value counts
df['gender'].value_counts()

<timed exec>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Wall time: 37.4 s


C:\Users\aengland\Anaconda3\lib\site-packages\pandas\core\frame.py:4308: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().drop(


male             869869
female           783945
unknown          390422
mostly_female     92403
mostly_male       73822
andy              14309
Name: gender, dtype: int64

In [15]:
# subset
df = df[df['gender'].isin(['male','female','mostly_female','mostly_male'])]

# show value counts
df['gender'].value_counts()

male             869869
female           783945
mostly_female     92403
mostly_male       73822
Name: gender, dtype: int64

In [16]:
# map
dict_map = {
    'male': 'male',
    'mostly_male': 'male',
    'female': 'female',
    'mostly_female': 'female',
}

# map target
df['gender'] = df['gender'].map(dict_map)

# show value counts
df['gender'].value_counts()

C:\Users\aengland\AppData\Local\Temp/ipykernel_17236/464687790.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['gender'] = df['gender'].map(dict_map)


male      943691
female    876348
Name: gender, dtype: int64

In [17]:
# show
df

,bigAccountId,bigDebtorId,strNameFirst,strNameLast,strZipCode,max_proba_race,race,gender
0,1338314,1757483,JOHN,DOE,84663,0.977112,white,male
1,1338520,1757748,RACHEL,COX,80112,0.976314,white,female
3,1338624,1757881,JOHN,DOE,63129,0.982650,white,male
4,1338702,1757981,PAMELA,BILLINGSLEA,30297,0.954750,black,female
5,1338756,1758048,SUSAN,SHAFFER,80232,0.990006,white,female
...,...,...,...,...,...,...,...,...
2224797,4254691,5447355,CONNIE,KIRK,74012,NaN,NaN,female
2224798,4254693,5447357,MARY,SMITH,31634,NaN,NaN,female
2224799,4254697,5447362,JOSHUA,GAMMON,43209,NaN,NaN,male
2224800,4254698,5447364,CHARLES,MORRIS,84660,NaN,NaN,male


### Join with raw data

In [18]:
%%time

# download
str_filename = 'df_raw.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'01_ad/01_data_prep/01_data_collection/output/{str_filename}'
download_from_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

# read
df_tmp = pd.read_parquet(str_local_path, columns=['uniqueid','bigaccountid__app','bigdebtorid__app','subjectage__ln'])
# rm
os.remove(str_local_path)

# show
df_tmp

Wall time: 1min 40s


,uniqueid,bigaccountid__app,bigdebtorid__app,subjectage__ln
0,3.967211e+14,NaN,NaN,38.0
1,3.968461e+14,NaN,NaN,31.0
2,3.968154e+14,NaN,NaN,36.0
3,3.960774e+14,NaN,NaN,-1.0
4,3.961559e+14,NaN,NaN,49.0
...,...,...,...,...
2100693,3.962406e+14,3962405.0,5096010.0,40.0
2100694,3.962423e+14,3962422.0,5096029.0,20.0
2100695,3.962481e+14,3962480.0,5096102.0,36.0
2100696,3.962504e+14,3962503.0,5096134.0,22.0


In [19]:
%%time

# join
df = pd.merge(
    left=df,
    right=df_tmp,
    left_on=['bigAccountId','bigDebtorId'],
    right_on=['bigaccountid__app','bigdebtorid__app'],
    how='inner',
)

# show
df

Wall time: 3.38 s


,bigAccountId,bigDebtorId,strNameFirst,strNameLast,strZipCode,max_proba_race,race,gender,uniqueid,bigaccountid__app,bigdebtorid__app,subjectage__ln
0,1338314,1757483,JOHN,DOE,84663,0.977112,white,male,1.338314e+14,1338314.0,1757483.0,23.0
1,1338520,1757748,RACHEL,COX,80112,0.976314,white,female,1.338520e+14,1338520.0,1757748.0,43.0
2,1338624,1757881,JOHN,DOE,63129,0.982650,white,male,1.338624e+14,1338624.0,1757881.0,63.0
3,1338702,1757981,PAMELA,BILLINGSLEA,30297,0.954750,black,female,1.338702e+14,1338702.0,1757981.0,42.0
4,1338756,1758048,SUSAN,SHAFFER,80232,0.990006,white,female,1.338756e+14,1338756.0,1758048.0,54.0
...,...,...,...,...,...,...,...,...,...,...,...,...
479337,3640873,4692226,JAMES,HARRISON,40108,0.982376,white,male,3.640873e+14,3640873.0,4692226.0,31.0
479338,3640880,4692235,EARL,FATHEREE,85324,0.672772,white,male,3.640880e+14,3640880.0,4692235.0,61.0
479339,3640882,4692237,TIMOTHY,EVANS,87114,0.926760,white,male,3.640882e+14,3640882.0,4692237.0,49.0
479340,3640888,4692244,SHAWN,GREENWELL,62451,0.977962,white,male,3.640888e+14,3640888.0,4692244.0,-1.0


### Write to ```.csv```

In [20]:
%%time

str_filename = 'df_race_gender_age.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 9.48 s


### Upload to s3

In [21]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'ad_hoc/compliance/{str_filename}', 
    str_bucket_name=str_project,
)

# rm csv
os.remove(str_local_path)

Wall time: 1.9 s
